<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/07_2_Chain_QA_Agent_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 07-2: Chain 기반 Q&A 에이전트 구현
이 노트북에서는 LangChain의 가장 핵심적인 단위인 **Chain**을 구성하고, 이를 통해 단순 질의응답(Q&A) 에이전트를 만드는 실습을 진행합니다.

### 1. 환경 준비
필요한 라이브러리를 설치합니다.

In [1]:
!pip install -q langchain langchain-community langchain-huggingface transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


### 2. LLM 로드
Hugging Face의 `google/flan-t5-base` 모델을 로드하여 LangChain과 연결합니다.

In [2]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# 로컬 Hugging Face 파이프라인 생성
hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1,  # CPU 사용
    max_new_tokens=256
)

# LangChain 인터페이스로 래핑
llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("✅ LLM 로드 완료")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ LLM 로드 완료


### 2-1. Gemini LLM 로드 (선택 사항)

기존 Hugging Face 모델 대신 Google Gemini 모델을 사용하여 더 나은 답변을 시도할 수 있습니다. Gemini 모델을 사용하려면 `google-generativeai` 라이브러리를 설치하고 API 키를 설정해야 합니다.

In [18]:
!pip install -q google-generativeai langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 16.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


Gemini API를 사용하려면 API 키가 필요합니다.

아직 키가 없다면 Google AI Studio에서 키를 생성하세요.

1. Google AI Studio 접속
먼저 공식 사이트(https://aistudio.google.com)에 접속합니다. 사용 중인 구글 계정으로 로그인해 주세요.

2. 서비스 약관 동의
처음 접속하신 경우, 생성형 AI 사용을 위한 서비스 약관 동의 팝업이 뜹니다. 내용을 확인하신 후 'Accept' 또는 'Continue' 버튼을 클릭하여 메인 대시보드로 진입합니다.

3. API 키 메뉴 이동  
    - [대시보드] 왼쪽 상단 메뉴 바에서 [Get API key] 항목을 클릭합니다.

4. API 키 생성: 화면 중앙에 보이는 버튼 중 하나를 선택합니다.  

    - [Create API key] in new project: 새로운 프로젝트를 생성하면서 키를 발급받습니다. (처음 만드시는 분들께 권장)

    - Create API key in existing project: 기존에 사용하던 Google Cloud 프로젝트가 있다면 해당 프로젝트를 선택하여 키를 생성합니다.

5. 키 복사 및 안전한 보관:  
팝업창에 생성된 **긴 문자열(API Key)**이 나타납니다. 'Copy' 버튼을 눌러 복사한 뒤, 메모장이나 환경 변수 설정 등 안전한 곳에 저장해 두세요.

    ⚠️ 주의: API 키는 비밀번호와 같습니다. GitHub 같은 공개 저장소에 코드를 올릴 때 키가 노출되지 않도록 주의하세요!

6. 팁: 요금 및 제한 사항 (무료 티어 기준)  

    - Gemini 1.5 Flash: 속도가 빠르고 무료 사용량이 넉넉하여 테스트용으로 좋습니다.  
    - Gemini 1.5 Pro: 복잡한 추론에 적합하지만, 무료 티어에서는 분당 요청 횟수(RPM) 제한이 더 타이트합니다.  
    - 개인정보: 무료 등급 사용 시 입력한 데이터는 모델 학습에 사용될 수 있으므로 민감한 정보는 입력하지 않는 것이 좋습니다.  

Colab에서는 왼쪽 패널의 "🔑" 아래에 키를 `GOOGLE_API_KEY`라는 이름으로 Secrets Manager에 추가하세요. 그런 다음 키를 SDK에 전달합니다.

In [19]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("✅ Gemini API 설정 완료")

✅ Gemini API 설정 완료


이제 Gemini 모델을 초기화하여 `llm` 변수에 할당하겠습니다. 이렇게 하면 기존 Hugging Face 모델 대신 Gemini 모델이 사용됩니다.

In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Gemini 모델 초기화
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

print("✅ Gemini LLM 로드 완료")

✅ Gemini LLM 로드 완료


이제 `qa_chain`은 새로 로드된 Gemini LLM을 사용하게 됩니다. 다시 질문을 실행하여 답변을 확인해보겠습니다.

### 3. PromptTemplate 정의
에이전트의 사고 규칙(역할, 스타일)을 설정합니다.

In [27]:
from langchain_core.prompts import PromptTemplate

# 에이전트 페르소나 및 지시문 설정
template = """너는 친절하고 똑똑한 AI 도우미야.
다음 질문에 대해 핵심 위주로 명확하게 답변해줘.

질문: {question}

답변:"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)
print("✅ PromptTemplate 구성 완료")

✅ PromptTemplate 구성 완료


### 2-2. 사용 가능한 Gemini 모델 목록 확인
현재 API에서 사용할 수 있는 Gemini 모델 목록을 확인하여 정확한 모델 이름을 파악합니다.

In [25]:
import google.generativeai as genai

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-p

### 4. Chain 구성 및 실행
Prompt와 LLM을 파이프(`|`) 연산자로 연결(LCEL 방식)합니다.

In [30]:
# Chain 구성 (Prompt | LLM)
qa_chain = prompt | llm

# 에이전트 실행 테스트
user_question = "LangChain의 Chain 구조를 사용하면 어떤 장점이 있어?"
response = qa_chain.invoke({"question": user_question})

print(f"질문: {user_question}")
print("-" * 30)
print(f"답변: {response}")

질문: LangChain의 Chain 구조를 사용하면 어떤 장점이 있어?
------------------------------
답변: content=[{'type': 'text', 'text': "LangChain의 `Chain` 구조는 복잡한 LLM(대규모 언어 모델) 애플리케이션을 효율적으로 설계하고 관리할 수 있게 해주는 핵심적인 도구입니다.\n\n다음은 Chain 구조를 사용할 때 얻을 수 있는 주요 장점들입니다.\n\n---\n\n### LangChain Chain 구조의 핵심 장점\n\n#### 1. 모듈화 및 재사용성 (Modularity and Reusability)\n\n*   **복잡성 분해:** 복잡한 LLM 작업을 여러 개의 작은 단계(모듈)로 명확하게 분리하여 관리할 수 있습니다.\n*   **쉬운 재활용:** 특정 단계를 한 번 정의하면 다른 애플리케이션에서도 쉽게 재사용할 수 있어 개발 효율성이 높아집니다.\n\n#### 2. 표준화된 통합 및 연결성 (Standardized Integration)\n\n*   **구성 요소 연결:** LLM, 프롬프트 템플릿, 메모리, 데이터베이스, 외부 도구(Tools) 등 다양한 LangChain 구성 요소를 통일된 방식으로 쉽게 연결하고 통합할 수 있습니다.\n*   **파이프라인 구축:** 각 구성 요소가 명확한 입출력을 갖는 '파이프라인' 형태로 작동하므로, 데이터 흐름을 예측하고 제어하기 용이합니다.\n\n#### 3. 입출력 관리 및 흐름 제어 (I/O Management and Flow Control)\n\n*   **자동화된 데이터 전달:** 한 단계의 출력이 다음 단계의 입력으로 자동 전달되는 과정을 체계적으로 관리하여, 개발자가 직접 데이터 전달 코드를 작성할 필요가 줄어듭니다.\n*   **디버깅 용이성:** 각 단계의 입력과 출력이 명확하게 분리되어 있어, 문제가 발생했을 때 오류 발생 지점을 빠르게 파악하고 디버깅할 수 있습니다.\n\n#### 4. 복잡한 워크플로우 구축

### 5. 학습 정리
- **Chain**은 Prompt, LLM, Output Parser를 연결하는 파이프라인입니다.
- 이 구조를 통해 프롬프트를 재사용하고 로직을 모듈화할 수 있습니다.
- 다음 차시에서는 여기에 **Memory**를 추가하여 이전 대화를 기억하게 만듭니다.